In [1]:
import accelforge as af
from scheduling.scheduler import *
from af_wrapper import *
import numpy as np


def run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = None,
    total_latency_grid = None,
    actions_grid = None,
    memory_name = None,
    shared_memory_info = None,
):
    schedule, min_latency = best_schedule(
        einsums,
        compute_units,
        shared_memory_info,
        data_dependencies,
        latency_per_component_grid,
        total_latency_grid,
        actions_grid,
        memory_name
    )
    return schedule, min_latency


In [2]:
# Toy example. This only works with the non-resource-aware scheduler,
# which takes a grid of latencies as inputs rather than mappings
toy_data_dependencies = {
    "e1": [],
    "e2": [],
    "e3": ["e1", "e2"]
}
toy_compute_units = ['slow', 'fast']
toy_einsums = toy_data_dependencies.keys()
toy_grid = {
    ('slow', 'e1'): 0.2,
    ('fast', 'e1'): 0.1,
    ('slow', 'e2'): 0.1,
    ('fast', 'e2'): 0.05,
    ('slow', 'e3'): 0.2,
    ('fast', 'e3'): 0.1,
}
run(
    toy_einsums,
    toy_compute_units,
    toy_data_dependencies,
    total_latency_grid=toy_grid)

({(e1, fast, latency=0.1): 0,
  (e2, slow, latency=0.1): 0,
  (e3, fast, latency=0.1): 0.1},
 0.2)

In [3]:
arch = "arch/eyeriss-submods/full.yaml"
m2_workload = "workload/milestone-2/full.yaml"

In [4]:
# m2_full_mapping = af_map(
#     "arch/eyeriss-submods/full.yaml",
#     "workload/milestone-2/m2-full.yaml"
# )

In [5]:
# mappings = []
# for i in range(5):
#     filename = trunc(arch) + "-" + trunc(m2_workload) + "-" + str(i)
#     with open("images/" + filename + ".svg", "w") as f:
#         f.write(m2_full_mapping[i].render())
#     mappings.append(process_mapping(arch, m2_workload, m2_full_mapping[i]))

# # m2_full_mapping.to_dict()
# mappings

In [6]:
data_dependencies = {
    "E0": [],
    "E1": [],
    "E2": ["E0"],
    "E3": ["E0", "E1"], # weirdly we get a cycle if we have a different dep graph where E3 not dep on E0
    "E4": ["E2", "E3"]
}
compute_units = ['slow', 'fast']
einsums = data_dependencies.keys()
memory_name = "MainMemory"
shared_mem_info = {'GlobalBuffer' : [(0, 100), (100, 0), (50, 50), (25, 75), (75, 25)]}

In [7]:
from arch.arch_utils import *

arch_pairings = generate_architecture_pairings(compute_units, shared_mem_info)


In [17]:
# RUN TO GENERATE ACCELFORGE VALUES FOR SHARED MEMORY-AWARE SCHEDULING.
# To skip recomputation when running with fresh kernel, comment out
# this cell and use the below assignments instead.

# (grid_lats, grid_mems, grid_maps) = af_memoizable_grid_mem(
#     einsums, 
#     arch_pairings,
#     lambda einsum: "workload/milestone-2/"+einsum+".yaml",
#     "arch/eyeriss-shared-mem/"
# )

In [11]:
# this is for bw-aware version
# RUN TO GENERATE ACCELFORGE VALUES.
# To skip recomputation when running with fresh kernel, comment out
# this cell and use the below assignments instead.

# (grid_lats, grid_mems, grid_maps) = af_memoizable_grid(
#     einsums, 
#     compute_units,
#     lambda einsum: "workload/milestone-2/"+einsum+".yaml",
#     lambda sub_arch: "arch/eyeriss-submods/"+sub_arch
# )

In [14]:
grid_lats = {(('slow', ('GlobalBuffer', 0)), 'E0'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(0.00024576),
  'SlowWeightScratchpad': np.float32(3.3454155e-05),
  'SlowOutputScratchpad': np.float32(0.00021865385)},
 (('slow', ('GlobalBuffer', 0)), 'E1'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 0)), 'E2'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(0.00024576),
  'SlowOutputScratchpad': np.float32(0.00021865385),
  'SlowWeightScratchpad': np.float32(3.3454155e-05)},
 (('slow', ('GlobalBuffer', 0)), 'E3'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 0)), 'E4'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'SlowOutputScratchpad': np.float32(1.4385121e-06),
  'SlowWeightScratchpad': np.float32(3.136327e-07)},
 (('slow', ('GlobalBuffer', 25)), 'E0'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowWeightScratchpad': np.float32(3.3454155e-05),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowOutputScratchpad': np.float32(0.00021865385)},
 (('slow', ('GlobalBuffer', 25)), 'E1'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 25)), 'E2'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowOutputScratchpad': np.float32(0.00021865385),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowWeightScratchpad': np.float32(3.3454155e-05)},
 (('slow', ('GlobalBuffer', 25)), 'E3'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 25)), 'E4'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowWeightScratchpad': np.float32(3.136327e-07)},
 (('slow', ('GlobalBuffer', 50)), 'E0'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowWeightScratchpad': np.float32(3.3454155e-05),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowOutputScratchpad': np.float32(0.00021865385)},
 (('slow', ('GlobalBuffer', 50)), 'E1'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 50)), 'E2'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowOutputScratchpad': np.float32(0.00021865385),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowWeightScratchpad': np.float32(3.3454155e-05)},
 (('slow', ('GlobalBuffer', 50)), 'E3'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 50)), 'E4'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowWeightScratchpad': np.float32(3.136327e-07)},
 (('slow', ('GlobalBuffer', 75)), 'E0'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowWeightScratchpad': np.float32(3.3454155e-05),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowOutputScratchpad': np.float32(0.00021865385)},
 (('slow', ('GlobalBuffer', 75)), 'E1'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 75)), 'E2'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowOutputScratchpad': np.float32(0.00021865385),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowWeightScratchpad': np.float32(3.3454155e-05)},
 (('slow', ('GlobalBuffer', 75)), 'E3'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 75)), 'E4'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowWeightScratchpad': np.float32(3.136327e-07)},
 (('slow', ('GlobalBuffer', 100)), 'E0'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowWeightScratchpad': np.float32(3.3454155e-05),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowOutputScratchpad': np.float32(0.00021865385)},
 (('slow', ('GlobalBuffer', 100)), 'E1'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 100)), 'E2'): {'SlowMAC': np.float32(0.00049152),
  'SlowInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'SlowOutputScratchpad': np.float32(0.00021865385),
  'GlobalBuffer': np.float32(4.16e-06),
  'SlowWeightScratchpad': np.float32(3.3454155e-05)},
 (('slow', ('GlobalBuffer', 100)), 'E3'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowOutputScratchpad': np.float32(1.4385121e-06)},
 (('slow', ('GlobalBuffer', 100)), 'E4'): {'SlowMAC': np.float32(3.84e-06),
  'SlowInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'SlowOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'SlowWeightScratchpad': np.float32(3.136327e-07)},
 (('fast', ('GlobalBuffer', 0)), 'E0'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(0.00024576),
  'FastWeightScratchpad': np.float32(3.3454155e-05),
  'FastOutputScratchpad': np.float32(0.00021865385)},
 (('fast', ('GlobalBuffer', 0)), 'E1'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 0)), 'E2'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(0.00024576),
  'FastOutputScratchpad': np.float32(0.00021865385),
  'FastWeightScratchpad': np.float32(3.3454155e-05)},
 (('fast', ('GlobalBuffer', 0)), 'E3'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 0)), 'E4'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.304e-05),
  'FastOutputScratchpad': np.float32(1.4385121e-06),
  'FastWeightScratchpad': np.float32(3.136327e-07)},
 (('fast', ('GlobalBuffer', 25)), 'E0'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastWeightScratchpad': np.float32(5.3526648e-05),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastOutputScratchpad': np.float32(0.00018412955)},
 (('fast', ('GlobalBuffer', 25)), 'E1'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 25)), 'E2'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastOutputScratchpad': np.float32(0.00018412955),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastWeightScratchpad': np.float32(5.3526648e-05)},
 (('fast', ('GlobalBuffer', 25)), 'E3'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 25)), 'E4'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastWeightScratchpad': np.float32(3.136327e-07)},
 (('fast', ('GlobalBuffer', 50)), 'E0'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastWeightScratchpad': np.float32(5.3526648e-05),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastOutputScratchpad': np.float32(0.00018412955)},
 (('fast', ('GlobalBuffer', 50)), 'E1'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 50)), 'E2'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastOutputScratchpad': np.float32(0.00018412955),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastWeightScratchpad': np.float32(5.3526648e-05)},
 (('fast', ('GlobalBuffer', 50)), 'E3'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 50)), 'E4'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastWeightScratchpad': np.float32(3.136327e-07)},
 (('fast', ('GlobalBuffer', 75)), 'E0'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastWeightScratchpad': np.float32(5.3526648e-05),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastOutputScratchpad': np.float32(0.00018412955)},
 (('fast', ('GlobalBuffer', 75)), 'E1'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 75)), 'E2'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastOutputScratchpad': np.float32(0.00018412955),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastWeightScratchpad': np.float32(5.3526648e-05)},
 (('fast', ('GlobalBuffer', 75)), 'E3'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 75)), 'E4'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastWeightScratchpad': np.float32(3.136327e-07)},
 (('fast', ('GlobalBuffer', 100)), 'E0'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastWeightScratchpad': np.float32(5.3526648e-05),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastOutputScratchpad': np.float32(0.00018412955)},
 (('fast', ('GlobalBuffer', 100)), 'E1'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 100)), 'E2'): {'FastMAC': np.float32(0.00016384),
  'FastInputScratchpad': np.float32(9.278403e-05),
  'MainMemory': np.float32(6.144e-05),
  'FastOutputScratchpad': np.float32(0.00018412955),
  'GlobalBuffer': np.float32(5.44e-06),
  'FastWeightScratchpad': np.float32(5.3526648e-05)},
 (('fast', ('GlobalBuffer', 100)), 'E3'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastWeightScratchpad': np.float32(3.136327e-07),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastOutputScratchpad': np.float32(1.4385121e-06)},
 (('fast', ('GlobalBuffer', 100)), 'E4'): {'FastMAC': np.float32(1.28e-06),
  'FastInputScratchpad': np.float32(1.4385121e-06),
  'MainMemory': np.float32(2.08e-05),
  'FastOutputScratchpad': np.float32(1.4385121e-06),
  'GlobalBuffer': np.float32(4.5e-08),
  'FastWeightScratchpad': np.float32(3.136327e-07)}}

In [16]:
grid_mems = {(('slow', ('GlobalBuffer', 0)),
  'E0'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad', 'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float32(1.048576e+06), ('MainMemory',
   'write'): np.float64(524288.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 0)),
  'E1'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 0)),
  'E2'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float32(1.048576e+06), ('MainMemory',
   'write'): np.float64(524288.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 0)),
  'E3'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 0)),
  'E4'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowMAC', 'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 25)),
  'E0'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 25)),
  'E1'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 25)),
  'E2'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 25)),
  'E3'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 25)),
  'E4'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowMAC', 'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 50)),
  'E0'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 50)),
  'E1'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 50)),
  'E2'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 50)),
  'E3'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 50)),
  'E4'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowMAC', 'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 75)),
  'E0'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 75)),
  'E1'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 75)),
  'E2'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 75)),
  'E3'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 75)),
  'E4'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowMAC', 'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 100)),
  'E0'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 100)),
  'E1'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 100)),
  'E2'): {('SlowInputScratchpad',
   'read'): np.float64(16777216.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('SlowOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('SlowOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('GlobalBuffer',
   'read'): np.float32(16384.0), ('GlobalBuffer',
   'write'): np.float32(10240.0), ('SlowWeightScratchpad',
   'read'): np.float64(16777216.0), ('SlowWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('SlowMAC',
   'compute'): np.float64(2097152.0)},
 (('slow', ('GlobalBuffer', 100)),
  'E3'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('SlowMAC',
   'compute'): np.float64(16384.0)},
 (('slow', ('GlobalBuffer', 100)),
  'E4'): {('SlowInputScratchpad',
   'read'): np.float64(131072.0), ('SlowInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('SlowOutputScratchpad',
   'read'): np.float32(131072.0), ('SlowOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('SlowWeightScratchpad',
   'read'): np.float64(131072.0), ('SlowWeightScratchpad',
   'write'): np.float32(65536.0), ('SlowMAC', 'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 0)),
  'E0'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float32(1.048576e+06), ('MainMemory',
   'write'): np.float64(524288.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('FastOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 0)),
  'E1'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 0)),
  'E2'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float32(1.048576e+06), ('MainMemory',
   'write'): np.float64(524288.0), ('FastOutputScratchpad',
   'read'): np.float32(1.9922944e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.9922944e+07), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(4.194304e+06), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 0)),
  'E3'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 0)),
  'E4'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(139264.0), ('MainMemory',
   'write'): np.float64(8192.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastMAC', 'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 25)),
  'E0'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 25)),
  'E1'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 25)),
  'E2'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 25)),
  'E3'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 25)),
  'E4'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastMAC', 'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 50)),
  'E0'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 50)),
  'E1'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 50)),
  'E2'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 50)),
  'E3'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 50)),
  'E4'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastMAC', 'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 75)),
  'E0'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 75)),
  'E1'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 75)),
  'E2'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 75)),
  'E3'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 75)),
  'E4'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastMAC', 'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 100)),
  'E0'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 100)),
  'E1'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 100)),
  'E2'): {('FastInputScratchpad',
   'read'): np.float64(16777216.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(262144.0), ('MainMemory',
   'write'): np.float64(131072.0), ('FastOutputScratchpad',
   'read'): np.float32(1.6777216e+07), ('FastOutputScratchpad',
   'write'): np.float32(1.6777216e+07), ('GlobalBuffer',
   'read'): np.float32(32768.0), ('GlobalBuffer',
   'write'): np.float32(2048.0), ('FastWeightScratchpad',
   'read'): np.float64(16777216.0), ('FastWeightScratchpad',
   'write'): np.float32(1.6777216e+07), ('FastMAC',
   'compute'): np.float64(2097152.0)},
 (('fast', ('GlobalBuffer', 100)),
  'E3'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('FastMAC',
   'compute'): np.float64(16384.0)},
 (('fast', ('GlobalBuffer', 100)),
  'E4'): {('FastInputScratchpad',
   'read'): np.float64(131072.0), ('FastInputScratchpad',
   'write'): np.float32(131072.0), ('MainMemory',
   'read'): np.float64(132096.0), ('MainMemory',
   'write'): np.float64(1024.0), ('FastOutputScratchpad',
   'read'): np.float32(131072.0), ('FastOutputScratchpad',
   'write'): np.float32(131072.0), ('GlobalBuffer',
   'read'): np.float32(144.0), ('GlobalBuffer',
   'write'): np.float32(144.0), ('FastWeightScratchpad',
   'read'): np.float64(131072.0), ('FastWeightScratchpad',
   'write'): np.float32(65536.0), ('FastMAC', 'compute'): np.float64(16384.0)}}

In [14]:
schedule, latency = run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = grid_lats,
    actions_grid = grid_mems,
    memory_name = memory_name,
    shared_memory_info = shared_mem_info
)

In [15]:
schedule

{(E0, ('fast', ('GlobalBuffer', 25)), latency=0.0001841295452322811): 0,
 (E2, ('fast', ('GlobalBuffer', 25)), latency=0.0001841295452322811): np.float64(0.0001841295452322811),
 (E1, ('slow', ('GlobalBuffer', 0)), latency=3.457788312669638e-05): 0,
 (E3, ('slow', ('GlobalBuffer', 0)), latency=3.457788312669638e-05): np.float64(0.0001841295452322811),
 (E4, ('slow', ('GlobalBuffer', 25)), latency=2.080000012938399e-05): np.float64(0.0003682590904645622)}

In [ ]:
latency

In [ ]:
mems = sum(
    sum(
        count
        for (action, count) in grid_mems[(node.compute_assignment, node.einsum_name)].items()
        if action[0] == memory_name
    ) 
    for node in schedule.keys()
)
mems

In [ ]:
# bwu
(mems/latency) / 6.4e9 # old schedule: np.float64(0.4762258443913986)

In [ ]:
# Testing architecture pairing gen
generate_architecture_pairings(
    ['c1', 'c2', 'c3'],
    {'m1' : [(1, 2, 3), (5, 10, 15)],
     'm2' : [(7, 14, 21), (11, 22, 33)]})